# Model Hyperparameter Optimization

## Objective

Optimize the most promising classification models using
GridSearchCV and cross-validation.

Selected models:
- Logistic Regression
- Support Vector Classifier (SVC)

Primary evaluation metric: Macro F1

All optimization will use training data only.
The independent test dataset will remain untouched.

In [1]:
from sklearn.model_selection import GridSearchCV
from sklearn.model_selection import StratifiedKFold
from sklearn.model_selection import cross_val_score

## 1. Data Preparation

Load the original dataset and reproduce the same train/test split
used during model development.

The test dataset will remain untouched during hyperparameter optimization.

In [7]:
import pandas as pd
from sklearn.model_selection import train_test_split


df = pd.read_csv("../data/raw/student_mental_health.csv")

X = df.drop(columns=['Student_ID', 'Mental_Health_Risk'])

y = df['Mental_Health_Risk']

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print("X_train:", X_train.shape)
print("X_test:", X_test.shape)
print("y_train:", y_train.shape)
print("y_test:", y_test.shape)

X_train: (1600, 15)
X_test: (400, 15)
y_train: (1600,)
y_test: (400,)


In [15]:
numerical_columns = X_train.select_dtypes(include="number").columns.tolist()

categorical_columns = X_train.select_dtypes(exclude="number").columns.tolist()

print("Numerical features:", numerical_columns)
print("Categorical features:", categorical_columns)

Numerical features: ['Age', 'Year_of_Study', 'Sleep_Hours', 'Study_Hours_Per_Week', 'GPA', 'Social_Media_Hours', 'Screen_Time_Hours', 'Financial_Stress', 'Family_Support', 'Attendance_Rate']
Categorical features: ['Gender', 'Physical_Activity', 'Part_Time_Job', 'Diet_Quality', 'Previous_Mental_Health_History']


In [16]:
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder

In [18]:
numerical_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

In [21]:
categorical_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OneHotEncoder(handle_unknown="ignore"))
])

In [26]:
preprocessor = ColumnTransformer(transformers=[
    ("num", numerical_pipeline, numerical_columns),
    ("cat", categorical_pipeline, categorical_columns)
])

In [27]:
print("Preprocessor created successfully!")
print(preprocessor)

Preprocessor created successfully!
ColumnTransformer(transformers=[('num',
                                 Pipeline(steps=[('imputer',
                                                  SimpleImputer(strategy='median')),
                                                 ('scaler', StandardScaler())]),
                                 ['Age', 'Year_of_Study', 'Sleep_Hours',
                                  'Study_Hours_Per_Week', 'GPA',
                                  'Social_Media_Hours', 'Screen_Time_Hours',
                                  'Financial_Stress', 'Family_Support',
                                  'Attendance_Rate']),
                                ('cat',
                                 Pipeline(steps=[('imputer',
                                                  SimpleImputer(strategy='most_frequent')),
                                                 ('encoder',
                                                  OneHotEncoder(handle_unknown='ignore'))]),
         

In [29]:
from sklearn.linear_model import LogisticRegression

In [31]:
logistic_pipeline = Pipeline([
    ("preprocessing", preprocessor),
    ("classifier", LogisticRegression(max_iter=1000, random_state=42))
])

cv = StratifiedKFold(
    n_splits=3,
    shuffle=True,
    random_state=42
)

baseline_f1_scores = cross_val_score(
    logistic_pipeline,
    X_train,
    y_train,
    cv=cv,
    scoring="f1_macro"
)


print("Baseline Macro F1 scores:", baseline_f1_scores)

print("Mean Macro F1:", baseline_f1_scores.mean())

print("Standard Deviation:", baseline_f1_scores.std())


Baseline Macro F1 scores: [0.73911447 0.75131849 0.76351688]
Mean Macro F1: 0.7513166136304238
Standard Deviation: 0.009962243051116731


In [40]:
param_grid = {
    "classifier__C": [0.01, 0.1, 1, 10, 100]
}

In [41]:
grid_search = GridSearchCV(
    estimator=logistic_pipeline,
    param_grid=param_grid,
    scoring="f1_macro",
    cv=cv,
    n_jobs=-1,
    return_train_score=True
)

grid_search.fit(X_train, y_train)

,"estimator estimator: estimator objectThis is assumed to implement the scikit-learn estimator interface.Either estimator needs to provide a ``score`` function,or ``scoring`` must be passed.",Pipeline(step...m_state=42))])
,"param_grid param_grid: dict or list of dictionariesDictionary with parameters names (`str`) as keys and lists ofparameter settings to try as values, or a list of suchdictionaries, in which case the grids spanned by each dictionaryin the list are explored. This enables searching over any sequenceof parameter settings.","{'classifier__C': [0.01, 0.1, ...]}"
,"scoring scoring: str, callable, list, tuple or dict, default=NoneStrategy to evaluate the performance of the cross-validated model onthe test set.If `scoring` represents a single score, one can use:- a single string (see :ref:`scoring_string_names`);- a callable (see :ref:`scoring_callable`) that returns a single value;- `None`, the `estimator`'s :ref:`default evaluation criterion <scoring_api_overview>` is used.If `scoring` represents multiple scores, one can use:- a list or tuple of unique strings;- a callable returning a dictionary where the keys are the metric names and the values are the metric scores;- a dictionary with metric names as keys and callables as values.See :ref:`multimetric_grid_search` for an example.",'f1_macro'
,"n_jobs n_jobs: int, default=NoneNumber of jobs to run in parallel.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary <n_jobs>`for more details... versionchanged:: v0.20 `n_jobs` default changed from 1 to None",-1
,"cv cv: int, cross-validation generator or an iterable, default=NoneDetermines the cross-validation splitting strategy.Possible inputs for cv are:- None, to use the default 5-fold cross validation,- integer, to specify the number of folds in a `(Stratified)KFold`,- :term:`CV splitter`,- an iterable yielding (train, test) splits as arrays of indices.For integer/None inputs, if the estimator is a classifier and ``y`` iseither binary or multiclass, :class:`StratifiedKFold` is used. In allother cases, :class:`KFold` is used. These splitters are instantiatedwith `shuffle=False` so the splits will be the same across calls.Refer :ref:`User Guide <cross_validation>` for the variouscross-validation strategies that can be used here... versionchanged:: 0.22 ``cv`` default value if None changed from 3-fold to 5-fold.",StratifiedKFo... shuffle=True)
,"return_train_score return_train_score: bool, default=FalseIf ``False``, the ``cv_results_`` attribute will not include trainingscores.Computing training scores is used to get insights on how differentparameter settings impact the overfitting/underfitting trade-off.However computing the scores on the training set can be computationallyexpensive and is not strictly required to select the parameters thatyield the best generalization performance... versionadded:: 0.19.. versionchanged:: 0.21 Default value was changed from ``True`` to ``False``",True
,"refit refit: bool, str, or callable, default=TrueRefit an estimator using the best found parameters on the wholedataset.For multiple metric evaluation, this needs to be a `str` denoting thescorer that would be used to find the best parameters for refittingthe estimator at the end.Where there are considerations other than maximum score inchoosing a best estimator, ``refit`` can be set to a function whichreturns the selected ``best_index_`` given ``cv_results_``. In thatcase, the ``best_estimator_`` and ``best_params_`` will be setaccording to the returned ``best_index_`` while the ``best_score_``attribute will not be available.The refitted estimator is made available at the ``best_estimator_``attribute and permits using ``predict`` directly on this``GridSearchCV`` instance.Also for multiple metric evaluation, the attributes ``best_index_``,``best_score_`` and ``best_params_`` will only be available if``refit`` is set and all of them will be determined w.r.t this spec

In [55]:
print("Best parameters:", grid_search.best_params_)

print("Best CV Macro F1:", grid_search.best_score_)

Best parameters: {'classifier__C': 1}
Best CV Macro F1: 0.7513166136304238


In [43]:
results_df = pd.DataFrame(grid_search.cv_results_)

columns_to_show = [
    "param_classifier__C",
    "mean_train_score",
    "mean_test_score",
    "std_test_score",
    "mean_fit_time"
]

display(
    results_df[columns_to_show].sort_values(
        by="mean_test_score",
        ascending=False
    )
)

,param_classifier__C,mean_train_score,mean_test_score,std_test_score,mean_fit_time
2,1.00,0.767311,0.751317,0.009962,0.128883
4,100.00,0.767681,0.749064,0.008643,0.083390
3,10.00,0.767671,0.748370,0.008484,0.108055
1,0.10,0.762404,0.738918,0.014482,0.105108
0,0.01,0.711731,0.683404,0.013703,0.085719


### Logistic Regression Optimization Results

GridSearchCV evaluated five regularization values:
C = [0.01, 0.1, 1, 10, 100].

**Results:**
- Best C: 1
- Best CV Macro F1: 0.7513
- Baseline CV Macro F1: 0.7513
- CV standard deviation: 0.00996

**Observations:**
- Hyperparameter tuning did not improve the baseline performance.
- Strong regularization (C=0.01) reduced both training and validation performance.
- Larger C values produced no meaningful validation improvement.
- The best configuration showed a relatively small training-validation gap.

**Decision:**
Keep C=1 as the preferred Logistic Regression configuration
among the values tested.

The independent test dataset was not used.

In [45]:
from sklearn.svm import SVC

In [46]:
svc_pipeline = Pipeline([
    ("preprocessing", preprocessor),
    ("classifier", SVC(kernel="rbf"))
])

In [49]:
svc_baseline_scores = cross_val_score(
    svc_pipeline,
    X_train,
    y_train,
    cv=cv,
    scoring="f1_macro"
)

print("SVC Baseline Macro F1:", svc_baseline_scores)

print("SVC Mean Macro F1:", svc_baseline_scores.mean())

print("SVC Standard Deviation:", svc_baseline_scores.std())

SVC Baseline Macro F1: [0.70286749 0.7129025  0.73877147]
SVC Mean Macro F1: 0.7181804885406583
SVC Standard Deviation: 0.015125402871374918


In [53]:
svc_param_grid = {
    "classifier__C": [0.1, 1, 10],
    "classifier__gamma": ["scale", 0.01, 0.1, 1]
}

In [54]:
svc_grid_search = GridSearchCV(
    estimator=svc_pipeline,
    param_grid=svc_param_grid,
    scoring="f1_macro",
    cv=cv,
    n_jobs=-1,
    return_train_score=True
)

svc_grid_search.fit(X_train, y_train)

,"estimator estimator: estimator objectThis is assumed to implement the scikit-learn estimator interface.Either estimator needs to provide a ``score`` function,or ``scoring`` must be passed.","Pipeline(step...ier', SVC())])"
,"param_grid param_grid: dict or list of dictionariesDictionary with parameters names (`str`) as keys and lists ofparameter settings to try as values, or a list of suchdictionaries, in which case the grids spanned by each dictionaryin the list are explored. This enables searching over any sequenceof parameter settings.","{'classifier__C': [0.1, 1, ...], 'classifier__gamma': ['scale', 0.01, ...]}"
,"scoring scoring: str, callable, list, tuple or dict, default=NoneStrategy to evaluate the performance of the cross-validated model onthe test set.If `scoring` represents a single score, one can use:- a single string (see :ref:`scoring_string_names`);- a callable (see :ref:`scoring_callable`) that returns a single value;- `None`, the `estimator`'s :ref:`default evaluation criterion <scoring_api_overview>` is used.If `scoring` represents multiple scores, one can use:- a list or tuple of unique strings;- a callable returning a dictionary where the keys are the metric names and the values are the metric scores;- a dictionary with metric names as keys and callables as values.See :ref:`multimetric_grid_search` for an example.",'f1_macro'
,"n_jobs n_jobs: int, default=NoneNumber of jobs to run in parallel.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary <n_jobs>`for more details... versionchanged:: v0.20 `n_jobs` default changed from 1 to None",-1
,"cv cv: int, cross-validation generator or an iterable, default=NoneDetermines the cross-validation splitting strategy.Possible inputs for cv are:- None, to use the default 5-fold cross validation,- integer, to specify the number of folds in a `(Stratified)KFold`,- :term:`CV splitter`,- an iterable yielding (train, test) splits as arrays of indices.For integer/None inputs, if the estimator is a classifier and ``y`` iseither binary or multiclass, :class:`StratifiedKFold` is used. In allother cases, :class:`KFold` is used. These splitters are instantiatedwith `shuffle=False` so the splits will be the same across calls.Refer :ref:`User Guide <cross_validation>` for the variouscross-validation strategies that can be used here... versionchanged:: 0.22 ``cv`` default value if None changed from 3-fold to 5-fold.",StratifiedKFo... shuffle=True)
,"return_train_score return_train_score: bool, default=FalseIf ``False``, the ``cv_results_`` attribute will not include trainingscores.Computing training scores is used to get insights on how differentparameter settings impact the overfitting/underfitting trade-off.However computing the scores on the training set can be computationallyexpensive and is not strictly required to select the parameters thatyield the best generalization performance... versionadded:: 0.19.. versionchanged:: 0.21 Default value was changed from ``True`` to ``False``",True
,"refit refit: bool, str, or callable, default=TrueRefit an estimator using the best found parameters on the wholedataset.For multiple metric evaluation, this needs to be a `str` denoting thescorer that would be used to find the best parameters for refittingthe estimator at the end.Where there are considerations other than maximum score inchoosing a best estimator, ``refit`` can be set to a function whichreturns the selected ``best_index_`` given ``cv_results_``. In thatcase, the ``best_estimator_`` and ``best_params_`` will be setaccording to the returned ``best_index_`` while the ``best_score_``attribute will not be available.The refitted estimator is made available at the ``best_estimator_``attribute and permits using ``predict`` directly on this``GridSearchCV`` instance.Also for multiple metric evaluation, the attributes ``best_index_``,``best_score_`` and ``best_params_`` will only be available if``refit`` is set and all 

In [57]:
print("Best SVC parameters:", svc_grid_search.best_params_)

print("Best SVC Macro F1:", svc_grid_search.best_score_)

Best SVC parameters: {'classifier__C': 10, 'classifier__gamma': 0.01}
Best SVC Macro F1: 0.7340528405203278


In [58]:
svc_results_df = pd.DataFrame(svc_grid_search.cv_results_)

In [63]:
svc_columns = [
    "param_classifier__C",
    "param_classifier__gamma",
    "mean_train_score",
    "mean_test_score",
    "std_test_score",
    "mean_fit_time"
]

display(
    svc_results_df[svc_columns]
    .sort_values(by="mean_test_score", ascending=False)
    .head(5)
)

,param_classifier__C,param_classifier__gamma,mean_train_score,mean_test_score,std_test_score,mean_fit_time
9,10.0,0.01,0.799514,0.734053,0.010716,0.150091
5,1.0,0.01,0.762110,0.719208,0.009981,0.196729
4,1.0,scale,0.848156,0.718180,0.015125,0.230764
6,1.0,0.1,0.884775,0.710013,0.003924,0.200799
8,10.0,scale,0.978387,0.675708,0.006570,0.228076


In [64]:
optimization_comparison = pd.DataFrame({
    "Model": ["Logistic Regression", "SVC"],
    
    "Baseline Macro F1": [
        baseline_f1_scores.mean(),
        svc_baseline_scores.mean()
    ],

    "Optimized Macro F1": [
        grid_search.best_score_,
        svc_grid_search.best_score_
    ],

    "CV Std": [
        grid_search.cv_results_["std_test_score"][grid_search.best_index_],
        svc_grid_search.cv_results_["std_test_score"][svc_grid_search.best_index_]
    ]
})

In [65]:
optimization_comparison["Improvement (pp)"] = (
    optimization_comparison["Optimized Macro F1"]
    - optimization_comparison["Baseline Macro F1"]
) * 100

In [66]:
display(optimization_comparison.round(4))

,Model,Baseline Macro F1,Optimized Macro F1,CV Std,Improvement (pp)
0,Logistic Regression,0.7513,0.7513,0.0100,0.0000
1,SVC,0.7182,0.7341,0.0107,1.5872


## Hyperparameter Optimization Conclusions

### 1. Methodology

Hyperparameter optimization was performed using GridSearchCV
with three-fold StratifiedKFold cross-validation.

The primary evaluation metric was Macro F1.

All experiments used the training dataset only (1,600 observations).

Preprocessing was included inside each model pipeline to prevent
data leakage during cross-validation.

No resampling was applied because the target classes are
approximately balanced.

### 2. Logistic Regression

The following C values were evaluated:

[0.01, 0.1, 1, 10, 100]

Results:
- Best C: 1
- Baseline Macro F1: 0.7513
- Optimized Macro F1: 0.7513
- CV standard deviation: 0.00996

The optimization did not improve the baseline model.

The default regularization value (C=1) performed best among
the configurations evaluated.

### 3. Support Vector Classifier

The following hyperparameters were evaluated:

- C: [0.1, 1, 10]
- Gamma: ["scale", 0.01, 0.1, 1]

Results:
- Best C: 10
- Best gamma: 0.01
- Baseline Macro F1: 0.7182
- Optimized Macro F1: 0.7341
- Improvement: 1.59 percentage points
- CV standard deviation: 0.01072

Optimization improved validation performance and reduced
the training-validation performance gap.

### 4. Computational Cost

- Logistic Regression: 15 cross-validation fits
- SVC: 36 cross-validation fits

Each search also refitted the best configuration on the full
training dataset.

SVC required more experiments because two hyperparameters
were optimized instead of one.

### 5. Model Selection

Logistic Regression remains the preferred candidate because:

- It achieved the highest cross-validation Macro F1.
- Its validation results were relatively stable.
- It showed a smaller training-validation performance gap.
- It is easier to interpret than an RBF SVC.

### 6. Limitations

- Hyperparameter searches explored a limited number of configurations.
- Results are based on three-fold cross-validation.
- Selecting hyperparameters using cross-validation can produce
  optimistic performance estimates.
- The independent test dataset has not yet been evaluated.
- The model is strictly educational and is not suitable
  for clinical diagnosis or real-world student assessment.

### Final Decision

Select Logistic Regression (C=1) as the leading candidate
for subsequent independent test evaluation.

Retain optimized SVC as a comparison model.

## Final Model Selection

Logistic Regression was selected as the final model based on
cross-validation performance.

Selected configuration:
- Model: Logistic Regression
- C: 1
- CV Macro F1: 0.7513

The complete fitted pipeline will be saved for independent
test evaluation and later use in Streamlit.

In [67]:
final_pipeline = grid_search.best_estimator_

print("Pipeline steps:", list(final_pipeline.named_steps.keys()))

print("Selected C:", final_pipeline.named_steps["classifier"].C)

Pipeline steps: ['preprocessing', 'classifier']
Selected C: 1


In [68]:
import joblib

model_path = "../models/final_logistic_regression_pipeline.joblib"

joblib.dump(final_pipeline, model_path)

print("Model saved successfully!")
print("Model path:", model_path)

Model saved successfully!
Model path: ../models/final_logistic_regression_pipeline.joblib
